# Tutorial 3: Tool Use with Claude

This tutorial teaches you how to extend Claude's capabilities with custom tools (function calling).

## What You'll Learn

- Understanding Claude's tool use capability
- Defining tools with JSON schemas
- Handling tool calls and responses
- Implementing multi-step tool interactions
- Error handling and validation
- Real-world tool use patterns

## What is Tool Use?

Tool use (also called function calling) allows Claude to:
- Execute external functions
- Access real-time data (APIs, databases)
- Perform calculations
- Interact with external systems

**Key Concept:** Claude doesn't execute tools directly. It returns structured requests for YOU to execute.

---

## Setup

In [ ]:
!pip install anthropic python-dotenv

In [ ]:
import os
import json
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()
client = Anthropic()

print("✓ Setup complete!")

## Example 1: Defining Your First Tool

Let's create a simple calculator tool:

In [ ]:
# Define a calculator tool
tools = [
    {
        "name": "calculator",
        "description": "Performs basic arithmetic operations. Supports addition, subtraction, multiplication, and division.",
        "input_schema": {
            "type": "object",
            "properties": {
                "operation": {
                    "type": "string",
                    "enum": ["add", "subtract", "multiply", "divide"],
                    "description": "The arithmetic operation to perform"
                },
                "a": {
                    "type": "number",
                    "description": "The first number"
                },
                "b": {
                    "type": "number",
                    "description": "The second number"
                }
            },
            "required": ["operation", "a", "b"]
        }
    }
]

print("Tool definition:")
print(json.dumps(tools[0], indent=2))

## Example 2: Making a Tool Use Request

Ask Claude to use the calculator:

In [ ]:
# Request that requires tool use
response = client.messages.create(
    model="claude-3-5-sonnet-20241022",
    max_tokens=1024,
    tools=tools,
    messages=[{
        "role": "user",
        "content": "What is 1,234 multiplied by 5,678?"
    }]
)

print("Response:")
print(f"Stop reason: {response.stop_reason}")
print(f"\nContent blocks: {len(response.content)}")

for block in response.content:
    print(f"\nBlock type: {block.type}")
    if block.type == "tool_use":
        print(f"Tool: {block.name}")
        print(f"Tool ID: {block.id}")
        print(f"Input: {json.dumps(block.input, indent=2)}")

### Understanding the Response

Key observations:
- `stop_reason` is `"tool_use"` when Claude wants to use a tool
- The response contains a `tool_use` content block
- Claude extracted the numbers (1234 and 5678) and operation (multiply)
- You must execute the tool and return results

## Example 3: Implementing Tool Execution

Now let's actually execute the tool:

In [ ]:
def execute_calculator(operation, a, b):
    """Execute calculator operations."""
    operations = {
        "add": lambda x, y: x + y,
        "subtract": lambda x, y: x - y,
        "multiply": lambda x, y: x * y,
        "divide": lambda x, y: x / y if y != 0 else "Error: Division by zero"
    }
    
    if operation not in operations:
        return {"error": f"Unknown operation: {operation}"}
    
    result = operations[operation](a, b)
    return {"result": result}

# Execute the tool from previous response
tool_use_block = next(block for block in response.content if block.type == "tool_use")
tool_result = execute_calculator(**tool_use_block.input)

print(f"Tool execution result: {tool_result}")

## Example 4: Returning Tool Results to Claude

Send the tool result back to Claude for a final response:

In [ ]:
# Continue conversation with tool result
final_response = client.messages.create(
    model="claude-3-5-sonnet-20241022",
    max_tokens=1024,
    tools=tools,
    messages=[
        {"role": "user", "content": "What is 1,234 multiplied by 5,678?"},
        {"role": "assistant", "content": response.content},
        {
            "role": "user",
            "content": [
                {
                    "type": "tool_result",
                    "tool_use_id": tool_use_block.id,
                    "content": json.dumps(tool_result)
                }
            ]
        }
    ]
)

print("Claude's final response:")
print(final_response.content[0].text)

## Example 5: Complete Tool Use Loop

Let's wrap this in a reusable function:

In [ ]:
def process_tool_call(user_message, tools, max_iterations=5):
    """Complete tool use loop with automatic execution."""
    
    messages = [{"role": "user", "content": user_message}]
    
    for iteration in range(max_iterations):
        response = client.messages.create(
            model="claude-3-5-sonnet-20241022",
            max_tokens=1024,
            tools=tools,
            messages=messages
        )
        
        print(f"\nIteration {iteration + 1}:")
        print(f"Stop reason: {response.stop_reason}")
        
        if response.stop_reason == "end_turn":
            # No more tool use, return final answer
            final_text = next((block.text for block in response.content if hasattr(block, "text")), None)
            return final_text
        
        elif response.stop_reason == "tool_use":
            # Add assistant's response to messages
            messages.append({"role": "assistant", "content": response.content})
            
            # Execute all tool calls
            tool_results = []
            
            for block in response.content:
                if block.type == "tool_use":
                    print(f"  Tool: {block.name}({block.input})")
                    
                    # Execute the calculator
                    result = execute_calculator(**block.input)
                    print(f"  Result: {result}")
                    
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": json.dumps(result)
                    })
            
            # Add tool results to messages
            messages.append({"role": "user", "content": tool_results})
    
    return "Max iterations reached"

# Test the complete loop
result = process_tool_call(
    "Calculate: (123 + 456) * 2",
    tools
)

print("\n" + "="*60)
print("Final answer:")
print(result)

## Example 6: Multiple Tools

Let's add more tools for weather and time:

In [ ]:
# Define multiple tools
multi_tools = [
    {
        "name": "get_weather",
        "description": "Get current weather for a location",
        "input_schema": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "City name, e.g., 'San Francisco'"
                }
            },
            "required": ["location"]
        }
    },
    {
        "name": "get_time",
        "description": "Get current time for a timezone",
        "input_schema": {
            "type": "object",
            "properties": {
                "timezone": {
                    "type": "string",
                    "description": "Timezone, e.g., 'America/New_York'"
                }
            },
            "required": ["timezone"]
        }
    },
    tools[0]  # Include calculator
]

# Mock implementations
def get_weather(location):
    """Mock weather API."""
    return {
        "location": location,
        "temperature": 72,
        "conditions": "Sunny",
        "humidity": 45
    }

def get_time(timezone):
    """Mock time API."""
    from datetime import datetime
    return {
        "timezone": timezone,
        "time": datetime.now().strftime("%H:%M:%S"),
        "date": datetime.now().strftime("%Y-%m-%d")
    }

# Tool dispatcher
tool_functions = {
    "calculator": execute_calculator,
    "get_weather": get_weather,
    "get_time": get_time
}

print(f"✓ Registered {len(multi_tools)} tools")

In [ ]:
# Enhanced tool processor
def process_with_tools(user_message, available_tools, max_iterations=5):
    """Process messages with multiple tools."""
    
    messages = [{"role": "user", "content": user_message}]
    
    for iteration in range(max_iterations):
        response = client.messages.create(
            model="claude-3-5-sonnet-20241022",
            max_tokens=1024,
            tools=available_tools,
            messages=messages
        )
        
        if response.stop_reason == "end_turn":
            final_text = next((block.text for block in response.content if hasattr(block, "text")), None)
            return final_text
        
        elif response.stop_reason == "tool_use":
            messages.append({"role": "assistant", "content": response.content})
            
            tool_results = []
            
            for block in response.content:
                if block.type == "tool_use":
                    print(f"🔧 Using tool: {block.name}")
                    print(f"   Input: {json.dumps(block.input)}")
                    
                    # Execute the appropriate tool
                    if block.name in tool_functions:
                        result = tool_functions[block.name](**block.input)
                        print(f"   Result: {json.dumps(result)}\n")
                        
                        tool_results.append({
                            "type": "tool_result",
                            "tool_use_id": block.id,
                            "content": json.dumps(result)
                        })
            
            messages.append({"role": "user", "content": tool_results})
    
    return "Max iterations reached"

# Test with multiple tools
result = process_with_tools(
    "What's the weather in San Francisco? Also, if it's 72°F, what is that in Celsius? Use the formula (F - 32) * 5/9",
    multi_tools
)

print("="*60)
print("Final response:")
print(result)

## Example 7: Error Handling in Tool Use

Handle errors gracefully:

In [ ]:
def safe_tool_execution(tool_name, tool_input):
    """Execute tool with error handling."""
    
    try:
        if tool_name not in tool_functions:
            return {
                "error": f"Unknown tool: {tool_name}",
                "available_tools": list(tool_functions.keys())
            }
        
        result = tool_functions[tool_name](**tool_input)
        return {"success": True, "data": result}
    
    except TypeError as e:
        return {
            "error": f"Invalid arguments: {str(e)}",
            "tool": tool_name,
            "provided_input": tool_input
        }
    
    except Exception as e:
        return {
            "error": f"Tool execution failed: {str(e)}",
            "tool": tool_name
        }

# Test error handling
print("Valid call:")
print(safe_tool_execution("calculator", {"operation": "add", "a": 5, "b": 3}))

print("\nInvalid tool:")
print(safe_tool_execution("nonexistent", {}))

print("\nMissing arguments:")
print(safe_tool_execution("calculator", {"operation": "add"}))

## Example 8: Tool Use with Streaming

Combine tool use with streaming for better UX:

In [ ]:
def stream_with_tools(user_message, available_tools):
    """Stream responses with tool use."""
    
    messages = [{"role": "user", "content": user_message}]
    
    with client.messages.stream(
        model="claude-3-5-sonnet-20241022",
        max_tokens=1024,
        tools=available_tools,
        messages=messages
    ) as stream:
        print("Streaming response:\n")
        
        for event in stream:
            if event.type == "content_block_start":
                if hasattr(event.content_block, 'type'):
                    if event.content_block.type == "tool_use":
                        print(f"\n🔧 [Tool: {event.content_block.name}]")
            
            elif event.type == "content_block_delta":
                if hasattr(event.delta, 'text'):
                    print(event.delta.text, end="", flush=True)
        
        print("\n")
        return stream.get_final_message()

# Test streaming with tools
response = stream_with_tools(
    "What is 15 + 27?",
    tools
)

## Example 9: Real-World Tool - Web Search

Let's create a more realistic tool:

In [ ]:
# Define a web search tool
search_tools = [
    {
        "name": "web_search",
        "description": "Search the web for current information. Use this when you need up-to-date information or facts you don't know.",
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query"
                },
                "num_results": {
                    "type": "integer",
                    "description": "Number of results to return",
                    "default": 3
                }
            },
            "required": ["query"]
        }
    }
]

def web_search(query, num_results=3):
    """Mock web search (in production, use real search API)."""
    return {
        "query": query,
        "results": [
            {
                "title": f"Result {i+1} for: {query}",
                "snippet": f"This is a mock search result snippet {i+1}...",
                "url": f"https://example.com/result{i+1}"
            }
            for i in range(num_results)
        ]
    }

# Test web search tool
response = client.messages.create(
    model="claude-3-5-sonnet-20241022",
    max_tokens=512,
    tools=search_tools,
    messages=[{
        "role": "user",
        "content": "Search for information about Python 3.12 new features"
    }]
)

print("Claude's tool use request:")
for block in response.content:
    if block.type == "tool_use":
        print(f"Tool: {block.name}")
        print(f"Input: {json.dumps(block.input, indent=2)}")

## Interactive Exercise: Build Your Own Tool

Create a custom tool for your use case:

In [ ]:
# YOUR TURN: Define a custom tool

my_custom_tool = [
    {
        "name": "todo_manager",  # Change this
        "description": "Manages a todo list",  # Change this
        "input_schema": {
            "type": "object",
            "properties": {
                "action": {
                    "type": "string",
                    "enum": ["add", "list", "complete"],
                    "description": "Action to perform"
                },
                "task": {
                    "type": "string",
                    "description": "Task description"
                }
            },
            "required": ["action"]
        }
    }
]

# Implement your tool
todo_list = []

def todo_manager(action, task=None):
    """YOUR IMPLEMENTATION HERE"""
    if action == "add" and task:
        todo_list.append({"task": task, "done": False})
        return {"status": "added", "task": task}
    
    elif action == "list":
        return {"todos": todo_list}
    
    elif action == "complete" and task:
        for item in todo_list:
            if item["task"] == task:
                item["done"] = True
        return {"status": "completed", "task": task}
    
    return {"error": "Invalid action or missing task"}

# Test your tool
print("Add task:", todo_manager("add", "Learn Claude API"))
print("List tasks:", todo_manager("list"))
print("Complete task:", todo_manager("complete", "Learn Claude API"))
print("List tasks:", todo_manager("list"))

## Production Best Practices

### 1. Validate Tool Inputs

```python
def validate_tool_input(tool_name, tool_input, schema):
    # Check required fields
    required = schema.get("required", [])
    for field in required:
        if field not in tool_input:
            raise ValueError(f"Missing required field: {field}")
    
    # Validate types
    properties = schema.get("properties", {})
    for field, value in tool_input.items():
        if field in properties:
            expected_type = properties[field].get("type")
            # Add type checking logic
```

### 2. Set Timeouts for Tool Execution

```python
import signal

def timeout_handler(signum, frame):
    raise TimeoutError("Tool execution timeout")

signal.signal(signal.SIGALRM, timeout_handler)
signal.alarm(5)  # 5 second timeout
```

### 3. Log Tool Usage

```python
import logging

logging.info(f"Tool called: {tool_name}")
logging.info(f"Input: {tool_input}")
logging.info(f"Result: {result}")
```

### 4. Rate Limit Tool Calls

```python
from collections import defaultdict
import time

tool_call_times = defaultdict(list)

def check_rate_limit(tool_name, max_calls=10, window=60):
    now = time.time()
    # Remove old calls
    tool_call_times[tool_name] = [
        t for t in tool_call_times[tool_name] 
        if now - t < window
    ]
    
    if len(tool_call_times[tool_name]) >= max_calls:
        raise RateLimitError(f"Too many calls to {tool_name}")
    
    tool_call_times[tool_name].append(now)
```

## Key Takeaways

1. **Tool Use Extends Claude**: Add real-time data, calculations, and external system access

2. **JSON Schema Definition**: Tools require clear schemas with types and descriptions

3. **Conversation Loop**: User → Claude (tool request) → Execute tool → Return result → Claude (final answer)

4. **Multiple Tools**: Claude can choose from multiple tools and make multiple calls

5. **Error Handling**: Always validate inputs and handle execution errors gracefully

6. **Streaming Compatible**: Tool use works with streaming for better UX

## Common Tool Use Patterns

- **Data retrieval**: Weather, stocks, news, search
- **Calculations**: Math, conversions, analysis
- **External APIs**: Databases, CRMs, payment systems
- **File operations**: Read, write, search files
- **System commands**: Execute scripts, manage resources

## Next Steps

- **Tutorial 4**: Learn conversation management and context
- **Tutorial 5**: Apply production patterns at scale

## Resources

- [Anthropic Tool Use Guide](https://docs.anthropic.com/en/docs/tool-use)
- [Tool Use Examples](https://github.com/anthropics/anthropic-cookbook/tree/main/tool_use)
- [JSON Schema Reference](https://json-schema.org/understanding-json-schema/)